In [3]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Training_data_5.csv to Training_data_5 (1).csv


In [5]:
df = pd.read_csv("Training_data_5.csv")

headline_col = df.columns[0]
label_col = df.columns[1]

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.replace('&nbsp;', ' ').replace('&amp;', '&')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df[headline_col].apply(clean_text)

print("Dataset size:", len(df))
print("Classes:", df[label_col].unique())

Dataset size: 93229
Classes: ['Science and Technology' 'Sports' 'World News' 'Business']


In [6]:
label_list = df[label_col].unique().tolist()
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

df['label_id'] = df[label_col].map(label_to_id)

print(label_to_id)

{'Science and Technology': 0, 'Sports': 1, 'World News': 2, 'Business': 3}


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label_id'],
    test_size=0.2,
    random_state=42,
    stratify=df['label_id']
)

In [8]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=100)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_vec, y_train)

    preds = model.predict(X_test_vec)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')

    results[name] = (acc, f1)

    print(f"{name} -> Accuracy: {acc:.4f}, F1: {f1:.4f}")


Training Logistic Regression...
Logistic Regression -> Accuracy: 0.9089, F1: 0.9085

Training Naive Bayes...
Naive Bayes -> Accuracy: 0.8886, F1: 0.8881

Training SVM...
SVM -> Accuracy: 0.9110, F1: 0.9106

Training Random Forest...
Random Forest -> Accuracy: 0.8983, F1: 0.8959


In [10]:
print("\n--- Model Comparison ---")

for name, (acc, f1) in results.items():
    print(f"{name}: Accuracy={acc:.4f}, F1={f1:.4f}")


--- Model Comparison ---
Logistic Regression: Accuracy=0.9089, F1=0.9085
Naive Bayes: Accuracy=0.8886, F1=0.8881
SVM: Accuracy=0.9110, F1=0.9106
Random Forest: Accuracy=0.8983, F1=0.8959


In [11]:
best_model_name = max(results, key=lambda x: results[x][1])
best_model = models[best_model_name]

print("Best Model:", best_model_name)

joblib.dump(best_model, "best_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")
joblib.dump(id_to_label, "id_to_label.pkl")

Best Model: SVM


['id_to_label.pkl']

In [12]:
def predict(text):
    model = joblib.load("best_model.pkl")
    vectorizer = joblib.load("vectorizer.pkl")
    label_map = joblib.load("id_to_label.pkl")

    text = clean_text(text)
    vec = vectorizer.transform([text])

    pred_id = model.predict(vec)[0]
    return label_map[pred_id]

In [13]:
samples = [
    "<b>NASA launches new Mars rover</b>",
    "<b>Stock market hits record high</b>",
    "<b>Climate summit held in Paris</b>"
]

for s in samples:
    print(clean_text(s), "->", predict(s))

NASA launches new Mars rover -> Science and Technology
Stock market hits record high -> Business
Climate summit held in Paris -> World News
